# 02 — Optimizer Constraints Deep Dive

**What you'll learn:**
- How every constraint type in `RosterConstraints` shapes the optimal lineup
- Compare Standard vs Superflex vs No-Kicker/DST formats
- How player locks (forced starts/sits) affect total projected points
- How team stack limits force lineup diversification
- How salary caps create knapsack-style trade-offs
- How to combine multiple constraints for league-specific rules

**Prerequisites:** FFPy installed, `matplotlib`, `seaborn`. No database needed.

## Setup

This notebook uses FFPy's optimizer and scoring modules. No database required —
we construct sample players in-memory.

```python
# Run this cell to install analysis deps (first time only)
# !uv sync --group analysis
```

## Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

_repo = Path.cwd()
while not (_repo / "pyproject.toml").exists() and _repo.parent != _repo:
    _repo = _repo.parent
sys.path.insert(0, str(_repo / "src"))

from ffpy.optimizer import (
    LineupOptimizer, LineupResult,
    Player, PlayerStatus, RosterConstraints,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


## Data: a Larger Player Pool

In [ ]:
def make_pool():
    """Return 30+ players across all positions with realistic projections."""
    # ── QB (6) ──
    qbs = [
        Player("Josh Allen", "QB", "BUF", 25.8, opponent="MIA"),
        Player("Patrick Mahomes", "QB", "KC", 24.5, opponent="DEN"),
        Player("Lamar Jackson", "QB", "BAL", 24.2, opponent="CLE"),
        Player("Jalen Hurts", "QB", "PHI", 23.1, opponent="NYG"),
        Player("Joe Burrow", "QB", "CIN", 22.0, opponent="PIT"),
        Player("Dak Prescott", "QB", "DAL", 21.5, opponent="WAS"),
    ]
    # ── RB (8) ──
    rbs = [
        Player("Christian McCaffrey", "RB", "SF", 22.4, opponent="ARI"),
        Player("Saquon Barkley", "RB", "PHI", 20.1, opponent="NYG"),
        Player("Bijan Robinson", "RB", "ATL", 19.2, opponent="NO"),
        Player("Austin Ekeler", "RB", "WAS", 18.7, opponent="DAL"),
        Player("James Cook", "RB", "BUF", 15.2, opponent="MIA"),
        Player("Derrick Henry", "RB", "BAL", 17.5, opponent="CLE"),
        Player("Isiah Pacheco", "RB", "KC", 13.8, opponent="DEN"),
        Player("Raheem Mostert", "RB", "MIA", 14.2, opponent="BUF"),
    ]
    # ── WR (8) ──
    wrs = [
        Player("Tyreek Hill", "WR", "MIA", 19.8, opponent="BUF"),
        Player("CeeDee Lamb", "WR", "DAL", 18.5, opponent="WAS"),
        Player("Justin Jefferson", "WR", "MIN", 18.2, opponent="CHI"),
        Player("Amon-Ra St. Brown", "WR", "DET", 16.9, opponent="MIN"),
        Player("Stefon Diggs", "WR", "HOU", 14.3, opponent="IND"),
        Player("Davante Adams", "WR", "LV", 14.8, opponent="LAC"),
        Player("Deebo Samuel", "WR", "SF", 15.5, opponent="ARI"),
        Player("Garrett Wilson", "WR", "NYJ", 13.2, opponent="NE"),
    ]
    # ── TE (4) ──
    tes = [
        Player("Travis Kelce", "TE", "KC", 16.5, opponent="DEN"),
        Player("Mark Andrews", "TE", "BAL", 14.2, opponent="CLE"),
        Player("George Kittle", "TE", "SF", 13.1, opponent="ARI"),
        Player("Sam LaPorta", "TE", "DET", 12.5, opponent="MIN"),
    ]
    # ── K (4) ──
    ks = [
        Player("Justin Tucker", "K", "BAL", 9.5, opponent="CLE"),
        Player("Harrison Butker", "K", "KC", 9.2, opponent="DEN"),
        Player("Brandon Aubrey", "K", "DAL", 8.8, opponent="WAS"),
        Player("Jake Moody", "K", "SF", 8.5, opponent="ARI"),
    ]
    # ── DST (4) ──
    dsts = [
        Player("49ers DST", "DST", "SF", 10.2, opponent="ARI"),
        Player("Ravens DST", "DST", "BAL", 9.5, opponent="CLE"),
        Player("Cowboys DST", "DST", "DAL", 8.8, opponent="WAS"),
        Player("Chiefs DST", "DST", "KC", 8.5, opponent="DEN"),
    ]
    return qbs + rbs + wrs + tes + ks + dsts

pool = make_pool()
print(f"Player pool: {len(pool)}")
df_pool = pd.DataFrame([
    {"name": p.name, "pos": p.position, "team": p.team, "proj": p.projected_points}
    for p in pool
])
df_pool.groupby("pos").agg(count=("name", "count"), avg_proj=("proj", "mean"))


### 1. Roster Format Comparison

The three built-in roster formats differ in which positions are required
and how FLEX works:

In [ ]:
formats = {
    "Standard":     RosterConstraints.standard(),
    "No K/DST":     RosterConstraints.no_kicker_dst(),
    "Superflex":    RosterConstraints.superflex(),
}

for name, c in formats.items():
    print(f"{name:>12}  {c}")


In [ ]:
results_fmt = {}
for fmt_name, constraints in formats.items():
    opt = LineupOptimizer(constraints)
    res = opt.optimize(pool)
    results_fmt[fmt_name] = res

# Summary table
rows = []
for fmt_name, res in results_fmt.items():
    rows.append({
        "Format": fmt_name,
        "Starters": len(res.starters),
        "Total Pts": round(res.total_points, 1),
        "Avg/Starter": round(res.total_points / len(res.starters), 1),
        "Solve (ms)": round(res.solve_time_ms, 1),
    })
print("=== Format Comparison ===")
pd.DataFrame(rows).to_string(index=False)

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (fmt_name, res) in enumerate(results_fmt.items()):
    by_pos = res.get_starters_by_position()
    ax = axes[idx]
    y = 0
    for pos in ["QB", "RB", "WR", "TE", "FLEX", "K", "DST"]:
        for pl in by_pos.get(pos, []):
            ax.barh(y, pl.projected_points, height=0.7)
            ax.text(pl.projected_points + 0.3, y, pl.name, va="center", fontsize=8)
            y += 1
    ax.set_yticks([])
    ax.set_xlabel("Projected Points")
    ax.set_title(f"{fmt_name} — {res.total_points:.1f} pts")
plt.tight_layout()
plt.show()


**Key observations:**
- **No K/DST** frees up two roster spots, typically adding an extra WR or RB
- **Superflex** allows a second QB, which often boosts total points since QBs
  have the highest raw projections
- Solve times are trivial (<50ms) at this problem size


### 2. Player Locks (Forced Starts / Sits)

What happens when you must start or bench a specific player?

In [ ]:
# Baseline: standard format, no locks
opt_free = LineupOptimizer(RosterConstraints.standard())
free_res = opt_free.optimize(pool)

# Constraint: force-start Josh Allen, force-sit Christian McCaffrey
locked = RosterConstraints.standard()
locked.locked_in = {"Josh Allen"}
locked.locked_out = {"Christian McCaffrey"}

opt_locked = LineupOptimizer(locked)
locked_res = opt_locked.optimize(pool)

print("=== Free Optimization ===")
print(opt_free.analyze_lineup(free_res))
print()

print("=== Josh Allen forced START, CMC forced SIT ===")
print(opt_locked.analyze_lineup(locked_res))
print(f"\nCost of locks: {free_res.total_points - locked_res.total_points:.1f} pts")


### 3. Injury Status Handling

Players with `INJURED`, `OUT`, `BYE`, or `LOCKED` status are excluded.
`QUESTIONABLE` players are still eligible.

In [ ]:
injured_pool = make_pool()
# Simulate some injuries
injured_pool[0]  = Player("Josh Allen", "QB", "BUF", 25.8, status=PlayerStatus.INJURED)
injured_pool[3]  = Player("Jalen Hurts", "QB", "PHI", 23.1, status=PlayerStatus.OUT)
injured_pool[14] = Player("Stefon Diggs", "WR", "HOU", 14.3, status=PlayerStatus.QUESTIONABLE)
injured_pool[20] = Player("Travis Kelce", "TE", "KC", 16.5, status=PlayerStatus.BYE)

opt_inj = LineupOptimizer(RosterConstraints.standard())
inj_res = opt_inj.optimize(injured_pool)

print("=== Injury Scenario ===")
print(opt_inj.analyze_lineup(inj_res))
print()
# Show excluded
excluded = [p for p in injured_pool if not p.is_available()]
print("Excluded from pool:")
for p in excluded:
    print(f"  • {p.name:25} {p.status.value}")


### 4. Team Stack Limits

Some leagues limit how many players you can start from a single NFL team.
This encourages diversification and reduces reliance on one team's performance.

In [ ]:
# Create a pool with heavy KC/SF concentration
stack_pool = make_pool()
# Add extra KC and SF players
stack_pool.extend([
    Player("Rashee Rice", "WR", "KC", 12.5),
    Player("Isiah Pacheco", "RB", "KC", 13.8),
    Player("Brandon Aiyuk", "WR", "SF", 12.0),
    Player("Brock Purdy", "QB", "SF", 18.5),
])

# No limit baseline
opt_nolimit = LineupOptimizer(RosterConstraints.standard())
nolimit_res = opt_nolimit.optimize(stack_pool)

# Max 2 per team
limited = RosterConstraints.standard()
limited.max_players_per_team = 2
opt_limited = LineupOptimizer(limited)
limited_res = opt_limited.optimize(stack_pool)

def team_distribution(result):
    dist = {}
    for p in result.starters:
        dist[p.team] = dist.get(p.team, 0) + 1
    return dist

print("=== No Stack Limit ===")
print(opt_nolimit.analyze_lineup(nolimit_res))
print(f"Team dist: {team_distribution(nolimit_res)}")
print()

print("=== Max 2 Players/Team ===")
print(opt_limited.analyze_lineup(limited_res))
print(f"Team dist: {team_distribution(limited_res)}")
print()
print(f"Cost of constraint: {nolimit_res.total_points - limited_res.total_points:.1f} pts")


### 5. Salary Cap (DFS-Style)

DraftKings-style salary cap adds a knapsack constraint. Let's use a bigger
synthetic pool and sweep the cap to see the trade-off curve.

In [ ]:
import warnings
from pulp import PULP_CBC_CMD, LpMaximize, LpProblem, LpVariable, lpSum, value

def make_dfs_pool(n_per_pos=None, seed=42):
    """Synthetic DFS-style players with correlated salary & projection."""
    rng = np.random.default_rng(seed)
    if n_per_pos is None:
        n_per_pos = {"QB": 20, "RB": 30, "WR": 45, "TE": 15, "DST": 10}

    sal_range = {"QB": (5000, 9500), "RB": (3500, 10000),
                 "WR": (3000, 9500), "TE": (2800, 7500), "DST": (2000, 4500)}
    proj_range = {"QB": (8, 28), "RB": (4, 26), "WR": (3, 24),
                  "TE": (2, 18), "DST": (2, 14)}

    rows = []
    for pos, n in n_per_pos.items():
        salaries = rng.uniform(*sal_range[pos], size=n)
        projs = rng.uniform(*proj_range[pos], size=n)
        lo, hi = sal_range[pos]
        projs += (salaries - lo) / (hi - lo) * 5  # correlation
        for i, (s, pr) in enumerate(zip(salaries, projs)):
            rows.append({
                "player": f"{pos}_{i:03d}", "position": pos,
                "team": f"T{rng.integers(1, 33):02d}",
                "salary": int(round(s / 100) * 100),
                "projected_points": round(float(pr), 2),
            })
    return pd.DataFrame(rows)

dfs_pool = make_dfs_pool()
print(f"{len(dfs_pool)} players, "
      f"salary range ${dfs_pool['salary'].min():,}-${dfs_pool['salary'].max():,}")

def solve_salary_cap(df, cap):
    prob = LpProblem("DFS_Cap_Sweep", LpMaximize)
    players = df["player"].tolist()
    proj = dict(zip(df["player"], df["projected_points"]))
    pos = dict(zip(df["player"], df["position"]))
    sal = dict(zip(df["player"], df["salary"]))
    x = {p: LpVariable(f"x_{i}", cat="Binary") for i, p in enumerate(players)}
    prob += lpSum(proj[p] * x[p] for p in players)
    prob += lpSum(sal[p] * x[p] for p in players) <= cap
    # Position constraints (DK classic: 1 QB, 2 RB, 3 WR, 1 TE, 1 FLEX, 1 DST)
    pos_req = {"QB": 1, "RB": 2, "WR": 3, "TE": 1, "DST": 1}
    flex = ["RB", "WR", "TE"]
    for position, count in pos_req.items():
        eligible = [p for p in players if pos[p] == position]
        if position in flex:
            prob += lpSum(x[p] for p in eligible) >= count
        else:
            prob += lpSum(x[p] for p in eligible) == count
    flex_elig = [p for p in players if pos[p] in flex]
    base = sum(pos_req.get(fp, 0) for fp in flex)
    prob += lpSum(x[p] for p in flex_elig) == base + 1  # +1 flex
    prob += lpSum(x[p] for p in players) == sum(pos_req.values()) + 1
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob.solve(PULP_CBC_CMD(msg=0))
    if prob.status != 1:
        return None, 0, 0
    starters = [p for p in players if x[p].varValue and x[p].varValue > 0.5]
    total_sal = sum(sal[p] for p in starters)
    return starters, value(prob.objective), total_sal

caps = [35000, 40000, 45000, 50000, 55000, 60000]
cap_results = []
for cap in caps:
    line, obj, used = solve_salary_cap(dfs_pool, cap)
    cap_results.append({"cap": cap, "used": used, "projected": round(obj, 1),
                        "efficiency": round(obj / used * 10000, 2) if used else 0})

df_cap = pd.DataFrame(cap_results)
print()
print("=== Salary Cap Sweep ===")
print(df_cap.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(df_cap["cap"], df_cap["projected"], "bo-", label="Projected Points")
ax1.set_xlabel("Salary Cap ($)")
ax1.set_ylabel("Projected Points", color="b")
ax1.tick_params(axis="y", labelcolor="b")

ax2 = ax1.twinx()
ax2.plot(df_cap["cap"], df_cap["efficiency"], "rs--", label="Pts / $10k")
ax2.set_ylabel("Efficiency (pts / $10k)", color="r")
ax2.tick_params(axis="y", labelcolor="r")

fig.suptitle("Salary Cap vs Projected Points & Efficiency")
fig.tight_layout()
plt.show()


The efficiency curve shows diminishing returns — the highest-value players
(pts per dollar) are mid-range, but the optimizer can only take 9, so
eventually adding salary room doesn't help much.


### 6. Combining Constraints

Real leagues often combine multiple rules. Let's simulate a specific league:
- Standard roster (QB, 2RB, 2WR, TE, FLEX, K, DST)
- Max 2 players per NFL team (anti-stacking)
- Locked-in: the commissioner's favorite player (just for fun)
- Locked-out: a player you're avoiding


In [ ]:
combined = RosterConstraints(
    positions={"QB": 1, "RB": 2, "WR": 2, "TE": 1, "K": 1, "DST": 1},
    flex_positions=["RB", "WR", "TE"],
    num_flex=1,
    max_players_per_team=2,
    locked_in={"Patrick Mahomes"},
    locked_out={"Raheem Mostert"},
)

opt_comb = LineupOptimizer(combined)
comb_res = opt_comb.optimize(pool)

# Compare to free
opt_free2 = LineupOptimizer(RosterConstraints.standard())
free_res2 = opt_free2.optimize(pool)

print("=== Free (no extra constraints) ===")
print(f"  Total: {free_res2.total_points:.1f}  Starters: {len(free_res2.starters)}")
print("=== Combined Constraints ===")
print(opt_comb.analyze_lineup(comb_res))
print(f"\nCost: {free_res2.total_points - comb_res.total_points:.1f} pts")


### Key Takeaways

1. **Format choice matters** — Superflex adds ~6-8 pts vs Standard by letting
   you start a 2nd QB. No-K/DST adds a skill-position slot.
2. **Locks cost points** — every constraint reduces the feasible region. The
   cost equals the difference between the constrained and unconstrained optimum.
3. **Stack limits diversify risk** — but they'll force lower-projected backups
   if your top players share a team.
4. **Salary cap creates a knapsack** — the best players aren't always the best
   values. Efficiency (pts/dollar) matters.
5. **Combined constraints interact** — the cost of multiple constraints is
   not simply additive; the solver finds the best feasible solution.


---
*Notebook generated by `scripts/generate_optimization_notebooks.py`.*
